# File Processing with Grok

Extracting, analyzing, and comparing information from various file formats is essential for automating workflows in business, research, and technical domains. This notebook provides a comprehensive, production-ready guide for using xAI's Grok API to process documents in multiple formats including text files, PDFs, Word documents, and more. You will learn how to:

- Extract text and metadata from various file formats
- Summarize documents and large files
- Perform Q&A over document content
- Extract structured data (tables, key-value pairs)
- Analyze document metadata
- Compare versions for change detection

Each feature is designed for real-world scenarios such as contracts, reports, invoices, and technical documentation. The notebook includes robust error handling, caching, parallel processing, and best practices for scalable document analysis.

## Table of Contents
- [Setup and Prerequisites](#setup-and-prerequisites)
- **File Processing Features**
    - [1. Basic Summarization](#1-basic-summarization): Extracts and summarizes the main content of a file, ideal for quick overviews of contracts, reports, or articles.
    - [2. Chunked Summarization for Large Files](#2-chunked-summarization-for-large-files): Splits large documents into manageable chunks, summarizes each, and combines results for comprehensive coverage.
    - [3. Q&A Over File Content](#3-qa-over-file-content): Enables targeted question-answering over document content, useful for extracting specific facts, parties, or terms.
    - [4. Structured Data Extraction from Files](#4-structured-data-extraction-from-files): Extracts tables, key-value pairs, and entities as JSON for automation and downstream analysis.
    - [5. File Metadata Extraction and Model Interpretation](#file-metadata-extraction-and-model-interpretation): Retrieves and interprets metadata (author, creation date, size) for audit, compliance, and document management.
    - [6. File Comparison / Change Detection](#file-comparison--change-detection): Compares two versions of a file to highlight added, removed, or modified content, supporting version control and regulatory review.


---

**Usage Example:**
- Summarize a contract document and extract key terms as JSON
- Ask Grok to find the main topic or parties involved
- Compare two versions of a technical report to highlight changes
- Extract tables from a document for automated processing

**Expected Output:**
- Concise summary text
- Q&A responses
- Structured JSON with key-value pairs or tables
- Metadata insights (author, creation date, file size)
- Change report between document versions


## Setup and Prerequisites

You’ll need an X Developer account, an X API key and API secret. You'll also need an xAI API key which you can grab over at the xAI [console](https://console.x.ai)

In [1]:
%pip install -q openai tweepy python-dotenv aiohttp async_lru oauthlib PyPDF2 python-docx tiktoken

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Load API key from .env and validate
from dotenv import load_dotenv
import os
from openai import OpenAI
import sys

load_dotenv()
api_key = os.getenv('XAI_API_KEY')
GROK_MODEL = "grok-4"
BASE_URL = "https://api.x.ai/v1"

if not api_key:
    raise ValueError("XAI_API_KEY not found in environment. Please create a .env file with your API key.")

try:
    client = OpenAI(base_url=BASE_URL, api_key=api_key)
except Exception as e:
    print(f"Failed to initialize OpenAI client: {e}")
    sys.exit(1)

In [2]:
# Caching mechanism for Grok API calls
import hashlib
import json
CACHE_DIR = '.cache'
os.makedirs(CACHE_DIR, exist_ok=True)

def cached_api_call(prompt, model, client):
    cache_key = hashlib.md5(f"{prompt}{model}".encode()).hexdigest()
    cache_file = os.path.join(CACHE_DIR, f"{cache_key}.json")
    if os.path.exists(cache_file):
        with open(cache_file, 'r') as f:
            return json.load(f)
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=model
    )
    result = response.choices[0].message.content
    with open(cache_file, 'w') as f:
        json.dump(result, f)
    return result

## 1. Basic Summarization
Extract and summarize text from various file formats with file existence check and input sanitization

In [3]:

from PyPDF2 import PdfReader
from docx import Document
import re

def sanitize_path(path):
    # Only allow relative paths, no special characters
    if not re.match(r'^[\w\-/\.]+$', path):
        raise ValueError(f"Invalid file path: {path}")
    return path

def extract_text_from_file(file_path):
    """
    Extract text from various file formats.
    Supports: .txt, .md, .pdf, .docx
    """
    ext = os.path.splitext(file_path)[1].lower()
    
    try:
        if ext == '.pdf':
            reader = PdfReader(file_path)
            text = ' '.join(page.extract_text() for page in reader.pages if page.extract_text())
        elif ext == '.docx':
            doc = Document(file_path)
            text = ' '.join(paragraph.text for paragraph in doc.paragraphs)
        elif ext in ['.txt', '.md', '.csv', '.json', '.xml', '.html']:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
        else:
            raise ValueError(f"Unsupported file format: {ext}")
        return text
    except Exception as e:
        print(f"Error reading file: {e}")
        return ""

file_path = sanitize_path('file/sample.pdf')  # Change to your file path
if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}")

text = extract_text_from_file(file_path)

# Summarize the first 4000 characters
if text:
    summary = cached_api_call(f'Summarize this document: {text[:4000]}', GROK_MODEL, client)
    print("Basic Summary:\n", summary)
else:
    print("No text extracted from file.")

Basic Summary:
 This document is an **Employee Information Report** that lists ten employee records.

Each record contains the employee's name, department, city, position, and salary.

Key points from the report include:

*   **Individuals:** There are records for six unique individuals: Evan Rogers, Dana Lee, Charlie Davis, Bob Smith, and Alice Johnson.
*   **Multiple Entries:** Some employees are listed multiple times with different roles, locations, and salaries. Dana Lee appears four times, while Evan Rogers and Bob Smith each appear twice.
*   **Departments:** Employees are from four departments: Engineering, Finance, HR, and Marketing.
*   **Locations:** The cities represented are Houston, Los Angeles, Phoenix, New York, and Chicago.
*   **Salary Range:** Salaries listed range from **$43,541** to **$117,941**.

The document also notes that it was "Generated automatically for testing structured data extraction."


In [4]:
# Send extracted text to Grok for summarization
response = client.chat.completions.create(
    messages=[{"role": "user", "content": f'Summarize this document: {text[:4000]}' }],
    model=GROK_MODEL
    # You can change model to grok-2-vision-latest if needed
    # max_tokens=1024, # Optional: control response length
    # temperature=0.7, # Optional: control creativity
    # stream=False # Optional: enable streaming
    # ... other OpenAI params as needed
)
print(response.choices[0].message.content)

This document is an **Employee Information Report** containing 10 records for 5 different employees. The report lists each employee's name, department, city, position, and salary.

Here is a summary of the key information:

*   **Total Records:** 10
*   **Unique Employees:** 5 (Dana Lee, Evan Rogers, Bob Smith, Alice Johnson, Charlie Davis)

**Breakdown by Employee:**

*   **Dana Lee** is listed four times with different roles in various locations:
    *   Finance Consultant in Los Angeles ($50,407)
    *   Finance Developer in Phoenix ($43,541)
    *   Engineering Analyst in New York ($117,647)
    *   Engineering Developer in Phoenix ($49,032)
*   **Evan Rogers** appears twice:
    *   Engineering Analyst in Houston ($117,941)
    *   HR Developer in Phoenix ($43,873)
*   **Bob Smith** is also listed twice:
    *   Marketing Designer in Chicago ($90,742)
    *   Engineering Manager in New York ($73,386)
*   **Alice Johnson** is an HR Designer in Chicago ($85,268).
*   **Charlie Davis

## Notes

**Note:** For large files, you may want to chunk the text and process it in parts. This example sends the first 4000 characters to avoid API limits.

## 2. Chunked Summarization for Large Files

For large files, it's best to split the text into smaller chunks so each Grok API call stays within token limits. Here, we break the document into 4000-character chunks and summarize each chunk separately.

In [6]:
# 2. Chunked Summarization: Parallel processing of file chunks with caching
from concurrent.futures import ThreadPoolExecutor, as_completed

chunk_size = 4000
chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

def summarize_chunk(chunk):
    prompt = f"Summarize this document chunk: {chunk}"
    return cached_api_call(prompt, GROK_MODEL, client)

summaries = [""] * len(chunks)
with ThreadPoolExecutor(max_workers=3) as executor:
    future_to_idx = {executor.submit(summarize_chunk, chunk): idx for idx, chunk in enumerate(chunks)}
    for future in as_completed(future_to_idx):
        idx = future_to_idx[future]
        try:
            result = future.result()
            summaries[idx] = result if result is not None else ""
            print(f"Summary for chunk {idx+1}:\n", summaries[idx], "\n---\n")
        except Exception as e:
            print(f"Chunk {idx+1} failed: {e}")

# Optionally, combine all summaries for a final summary
if len(summaries) > 1:
    final_prompt = "Combine these summaries into a single summary: " + "\n".join(summaries)
    final_summary = cached_api_call(final_prompt, GROK_MODEL, client)
    print("\nFinal combined summary:\n", final_summary)

Summary for chunk 1:
 This document is an **Employee Information Report** that lists employees along with their department, city, position, and salary.

Key details include:
*   **Employees listed:** Evan Rogers, Dana Lee, Charlie Davis, Bob Smith, and Alice Johnson.
*   **Departments:** Engineering, Finance, HR, and Marketing.
*   **Locations:** Houston, Los Angeles, Phoenix, New York, and Chicago.
*   **Note:** Some employees, such as Dana Lee (four entries), Evan Rogers (two entries), and Bob Smith (two entries), appear multiple times with different roles and salaries.

The document also states it was **"Generated automatically for testing structured data extraction."** 
---



In [7]:
# Utility: Rate limiting, retries, and token counting for smart chunking
import time
import tiktoken
from functools import wraps

def rate_limited(max_per_second):
    min_interval = 1.0 / float(max_per_second)
    def decorator(func):
        last_time = [0.0]
        @wraps(func)
        def wrapper(*args, **kwargs):
            elapsed = time.time() - last_time[0]
            left_to_wait = min_interval - elapsed
            if left_to_wait > 0:
                time.sleep(left_to_wait)
            ret = func(*args, **kwargs)
            last_time[0] = time.time()
            return ret
        return wrapper
    return decorator

@rate_limited(2)  # Limit to 2 requests per second
def safe_cached_api_call(prompt, model, client, retries=3):
    for attempt in range(retries):
        try:
            return cached_api_call(prompt, model, client)
        except Exception as e:
            print(f"API call failed (attempt {attempt+1}): {e}")
            time.sleep(2 ** attempt)
    raise RuntimeError("API call failed after retries.")

def smart_chunk_text(text, max_tokens=2000):
    enc = tiktoken.encoding_for_model(GROK_MODEL)
    tokens = enc.encode(text)
    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i:i+max_tokens]
        chunk_text = enc.decode(chunk_tokens)
        chunks.append(chunk_text)
    return chunks

## 3. Q&A Over File Content

You can ask Grok questions about the file content. For best results, search for the most relevant chunk and send it with your question.

In [8]:
import time

def find_relevant_chunk(chunks, question):
    # Simple keyword search for demonstration; can be replaced with semantic search
    keywords = set(question.lower().split())
    best_idx = 0
    best_score = 0
    for idx, chunk in enumerate(chunks):
        score = sum(1 for word in keywords if word in chunk.lower())
        if score > best_score:
            best_score = score
            best_idx = idx
    return best_idx


question = "What is the main topic of this document?"  # Change as needed

relevant_chunk = []
if not chunks:
    print("No chunks found.")
else:
    relevant_idx = find_relevant_chunk(chunks, question)
    print(f"Selected chunk {relevant_idx + 1} of {len(chunks)} for Q&A.")

    relevant_chunk = chunks[relevant_idx]

    print("Sending Q&A request to Grok...")
    start_time = time.time()
    qa_response = cached_api_call(
        f"Answer this question about the document: {question}\n\nContent: {relevant_chunk}",
        GROK_MODEL,
        client
    )
    elapsed = time.time() - start_time
    print(f"Q&A Response (in {elapsed:.2f}s):\n", qa_response)

Selected chunk 1 of 1 for Q&A.
Sending Q&A request to Grok...
Q&A Response (in 5.24s):
 Based on the content, the main topic of this document is an **Employee Information Report**.

It lists employees and provides details about them, including their name, department, city, position, and salary.


## 4. Structured Data Extraction from Files

Grok can also extract structured data from your files, such as key-value pairs, tables, or named entities. This is useful for automating data entry or analysis tasks.

In [9]:
import json

# 4. Structured Data Extraction from Files: Extract key information as JSON with parsing validation
structured_prompt = (
    "Extract all key information from this document chunk and return it as a pure JSON object. "
    "Do NOT include ```json or ``` or any extra text. "
    "If there are tables, extract them as lists of dictionaries."
)

raw_response = safe_cached_api_call(
    f"{structured_prompt}\n\nContent: {relevant_chunk}",
    GROK_MODEL,
    client
)

print("Raw Structured Extraction Result:\n", raw_response)

# Clean up accidental formatting (if model added ```json etc.)
clean_response = (
    raw_response.replace("```json", "")
                .replace("```", "")
                .strip()
)

# Try to parse as JSON
try:
    structured_data = json.loads(clean_response)
    print("Parsed Structured Data:")
    print(structured_data)
except Exception as e:
    print("Failed to parse structured extraction as JSON:", e)

Raw Structured Extraction Result:
 ```json
{
  "report_title": "Employee Information Report",
  "employees": [
    {
      "Name": "Evan Rogers",
      "Department": "Engineering",
      "City": "Houston",
      "Position": "Analyst",
      "Salary": "$117,941"
    },
    {
      "Name": "Dana Lee",
      "Department": "Finance",
      "City": "Los Angeles",
      "Position": "Consultant",
      "Salary": "$50,407"
    },
    {
      "Name": "Dana Lee",
      "Department": "Finance",
      "City": "Phoenix",
      "Position": "Developer",
      "Salary": "$43,541"
    },
    {
      "Name": "Charlie Davis",
      "Department": "HR",
      "City": "Phoenix",
      "Position": "Designer",
      "Salary": "$84,698"
    },
    {
      "Name": "Dana Lee",
      "Department": "Engineering",
      "City": "New York",
      "Position": "Analyst",
      "Salary": "$117,647"
    },
    {
      "Name": "Bob Smith",
      "Department": "Marketing",
      "City": "Chicago",
      "Position": "Desig

## File Metadata Extraction and Model Interpretation

You can extract metadata from various file formats, such as author, creation date, modification date, file size, and more. Then, use Grok to interpret and summarize what the metadata reveals about the document. This is useful for document management, audit trails, and gaining insights from file properties.

In [10]:
# Extract and display file metadata
import os
from datetime import datetime

def get_file_metadata(file_path):
    """Extract metadata from various file formats"""
    metadata = {}
    
    # Get basic file stats
    stats = os.stat(file_path)
    metadata['File Size (bytes)'] = stats.st_size
    metadata['File Size (KB)'] = round(stats.st_size / 1024, 2)
    metadata['Created'] = datetime.fromtimestamp(stats.st_ctime).strftime('%Y-%m-%d %H:%M:%S')
    metadata['Modified'] = datetime.fromtimestamp(stats.st_mtime).strftime('%Y-%m-%d %H:%M:%S')
    metadata['File Extension'] = os.path.splitext(file_path)[1]
    
    # Get format-specific metadata
    ext = os.path.splitext(file_path)[1].lower()
    
    if ext == '.pdf':
        try:
            reader = PdfReader(file_path)
            metadata['Number of Pages'] = len(reader.pages)
            if reader.metadata:
                for key, value in reader.metadata.items():
                    metadata[key] = str(value)
        except:
            pass
    elif ext == '.docx':
        try:
            doc = Document(file_path)
            metadata['Number of Paragraphs'] = len(doc.paragraphs)
            metadata['Number of Sections'] = len(doc.sections)
            # Extract core properties if available
            if hasattr(doc, 'core_properties'):
                props = doc.core_properties
                if props.author:
                    metadata['Author'] = props.author
                if props.title:
                    metadata['Title'] = props.title
                if props.subject:
                    metadata['Subject'] = props.subject
        except:
            pass
    
    return metadata

metadata = get_file_metadata(file_path)

print("File Metadata:")
for key, value in metadata.items():
    print(f"{key}: {value}")

File Metadata:
File Size (bytes): 2353
File Size (KB): 2.3
Created: 2025-10-17 21:25:10
Modified: 2025-10-17 20:48:42
File Extension: .pdf
Number of Pages: 1
/Author: anonymous
/CreationDate: D:20251017151829+00'00'
/Creator: ReportLab PDF Library - www.reportlab.com
/Keywords: 
/ModDate: D:20251017151829+00'00'
/Producer: ReportLab PDF Library - www.reportlab.com
/Subject: unspecified
/Title: untitled
/Trapped: /False


In [11]:
import json

metadata_str = json.dumps(metadata, indent=2)

metadata_summary_response = client.chat.completions.create(
    messages=[
        {"role": "user", "content": f"Here is the metadata for a file. Summarize what you can infer about this document from its metadata, and highlight anything interesting or unusual.\n\nMetadata:\n{metadata_str}"}
    ],
    model=GROK_MODEL
)
print("Grok Metadata Summary:\n", metadata_summary_response.choices[0].message.content)

Grok Metadata Summary:
 Of course. Based on the provided metadata, here is a summary of what can be inferred about the document, with interesting and unusual points highlighted.

### Summary of Inferences

This is a very small, single-page PDF document. Its characteristics strongly suggest it was **programmatically generated by an automated process or script**, rather than being created manually by a person.

Here's the evidence for this conclusion:

*   **Creator/Producer:** The document was created using the "ReportLab PDF Library," a popular tool (specifically, a Python library) for generating PDFs automatically from code. This is a clear indicator that it was made by a script or application.
*   **Minimalist Content:** At only 2.3 KB, the file is tiny. This implies it contains very little content—likely just a few lines of text or a very simple vector graphic. It almost certainly does not contain any images.
*   **Generic Information:** The descriptive fields are all default or emp

## File Comparison / Change Detection

Compare two versions of a file to detect meaningful differences. This is useful for tracking changes in contracts, regulatory filings, or technical documentation. The workflow extracts text from both files and uses Grok to highlight added, removed, or modified content.

In [12]:
# Compare two files for changes using Grok with improved logic and error handling
def extract_file_text(path):
    """Extract text from a file using the extract_text_from_file function"""
    try:
        return extract_text_from_file(path)
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return ""

file_path_v1 = sanitize_path('file/sample_v2.pdf')  # First version
file_path_v2 = sanitize_path('file/sample.pdf')  # Second version

if not os.path.exists(file_path_v1) or not os.path.exists(file_path_v2):
    raise FileNotFoundError("One or both files for comparison not found.")

text_v1 = extract_file_text(file_path_v1)
text_v2 = extract_file_text(file_path_v2)

if not text_v1 or not text_v2:
    print("One or both files have no extractable text.")

compare_prompt = (
    "You are a document comparison assistant. Compare the following two versions of a file. "
    "Highlight any added, removed, or modified content. "
    "Be concise and focus on meaningful changes.\n\n"
    "--- Version 1 ---\n" + text_v1[:4000] +
    "\n--- Version 2 ---\n" + text_v2[:4000]
    )

compare_result = safe_cached_api_call(compare_prompt, GROK_MODEL, client)
print("File Comparison Result:\n", compare_result)

File Comparison Result:
 Here is a comparison of the two document versions.

The entire set of employee records has been replaced with a new list. There are no overlapping employees between the two versions.

### Summary of Changes

**Modified**
*   **Title:** Changed from "Updated Employee Information Report" to "Employee Information Report".
*   **Footer:** The purpose statement was changed from "Generated automatically for comparison testing" to "Generated automatically for testing structured data extraction".

**Removed**
*   All 10 employee records from Version 1 were removed, including entries for Clara Wilson, Ava Martinez, and Ella Anderson.

**Added**
*   10 completely new employee records were added, including entries for Evan Rogers, Dana Lee, Charlie Davis, Bob Smith, and Alice Johnson.


## Conclusion

This guide demonstrated how to process various file formats using Grok, covering:

- **Text Extraction:** Efficiently reading and extracting text from multiple file formats including PDFs, Word documents, text files, and more.
- **Summarization:** Leveraging Grok to generate concise summaries of document content, both in single and chunked formats for large files.
- **Q&A:** Asking targeted questions about file content and retrieving answers using Grok's language understanding.
- **Structured Data Extraction:** Automatically extracting key-value pairs, tables, and entities from documents for downstream automation.
- **Metadata Analysis:** Interpreting file metadata and using Grok to provide insights about document properties.
- **Change Detection:** Comparing different versions of files to highlight meaningful changes using Grok.

These features enable robust document analysis workflows, making it easy to automate information extraction, review, and comparison tasks with minimal code across various file formats.

## Next Steps

- Integrate with real-world document pipelines (e.g., contracts, reports, invoices).
- Expand structured extraction to support more complex tables and forms.
- Add error handling for corrupted or scanned files.
- Add support for additional file formats (e.g., Excel, CSV parsing with structure).
- Explore combining text and image analysis for multimodal documents.
- Contribute new file processing recipes to the xai-cookbook!